# Инженерная реализация MVP системы мониторинга состояния водителя

## Разработка и исследование MVP-ядра системы мониторинга состояния водителя для коммерческого транспорта

**Автор:** Волков Д.В.

**Роль в проекте:** разработка программного модуля (MVP) системы мониторинга состояния водителя, реализующего адаптивный пороговый подход и политопную фильтрацию событий.

---

## 1. Зависимость от научной части (Волкова Н.В.)

Система реализует научно обоснованные результаты, полученные Волковой Н.В.:

| Результат | Значение | Формат передачи |
|:---|:---|:---|
| K_MAR | 7.26 | `scientific_coefficients.json` |
| K_nose_chin | 1.38 | `scientific_coefficients.json` |
| K_EAR | 0.6 | литература (Soukupová & Čech, 2016) |
| Окно зевка | 15 кадров (0.5 с) | Техническое задание |
| Окно сонливости/отвлечения/наклона | 60 кадров (2.0 с) | Техническое задание |
| Порог отвлечения (\|yaw\|) | 30° | Техническое задание |
| Порог наклона (\|roll\|) | 20° | Техническое задание |
| Pitch | исключён (нестабильность solvePnP) | Техническое задание |

**Адаптивный пороговый подход:**

Порог_водителя = Базовый_уровень_водителя × K


где базовый уровень измеряется в ходе 30-секундной калибровки.

Научное обоснование выбора признаков, статистический анализ и ML-верификация выполнены в рамках отдельного исследования и **не входят** в состав данного программного модуля.

---

## 2. Функциональные возможности MVP

### 2.1. Четыре системы детекции

| Детектируемое состояние | Метрика | Тип политопа | Окно | Пороговое условие |
|:---|:---|:---|:---|:---|
| **Зевок** | MAR + Nose-Chin | 2D | 15 кадров (0.5 с) | MAR ≥ MAR_баз × 7.26 И Nose-Chin ≥ Nose-Chin_баз × 1.38 |
| **Сонливость** | EAR | 1D (below) | 60 кадров (2.0 с) | EAR < EAR_баз × 0.6 |
| **Отвлечение внимания** | \|Yaw\| | 1D (above) | 60 кадров (2.0 с) | \|Yaw\| ≥ 30° |
| **Наклон головы** | \|Roll\| | 1D (above) | 60 кадров (2.0 с) | \|Roll\| ≥ 20° |

### 2.2. PERCLOS

Метрика PERCLOS (Percentage of Eyelid Closure) рассчитывается в скользящем окне 30 секунд в соответствии со стандартом ISO/TR 21926.

| Уровень усталости | PERCLOS | Интерпретация |
|:---|:---|:---|
| normal | < 10% | Нормальное состояние |
| light | 10% – 20% | Начальная стадия усталости |
| medium | 20% – 28% | Выраженная усталость |
| critical | ≥ 28% (подтверждено 2 сек) | Критическая усталость (микросон) |

### 2.3. Интегральная оценка критичности

| Уровень | Цвет | Условие |
|:---|:---|:---|
| **NORMAL** | Зеленый | Нет событий |
| **WARNING** | Желтый | Зевок ИЛИ отвлечение |
| **DANGER** | Оранжевый | Сонливость ИЛИ наклон |
| **CRITICAL** | Красный | PERCLOS ≥ 28% ИЛИ два события одновременно |

---

## 3. Архитектура системы

### 3.1. Компонентная диаграмма

┌─────────────────────────────────────────────────────────────────────────────┐
│ FatigueMonitoringSystem │
├─────────────────────────────────────────────────────────────────────────────┤
│ │
│ ┌──────────────┐ ┌───────────────┐ ┌──────────────┐ │
│ │ VideoCapture │ ─▶ │ FaceMeshProc │ ─▶ │ MetricsCalc │ │
│ │ (OpenCV) │ │ (MediaPipe) │ │ (NumPy) │ │
│ └──────────────┘ └───────────────┘ └──────┬───────┘ │
│ │ │
│ ▼ │
│ ┌──────────────┐ ┌──────────────┐ ┌───────────────┐ │
│ │ ReportGen │ ◀── │ DatabaseMgr │ ◀── │FatigueAnalyzer│ │
│ │pandas,matplot│ │ (SQLite) │ │ (Polytopes) │ │
│ └──────────────┘ └──────────────┘ └───────┬───────┘ │
│ │ │
│ ▼ │
│ ┌──────────────┐ │
│ │ Calibrator │ │
│ │ Visualizer │ │
│ └──────────────┘ │
└─────────────────────────────────────────────────────────────────────────────┘


### 3.2. Ключевые классы

| Класс | Модуль | Назначение |
|:---|:---|:---|
| `FaceMetricsExtractor` | `extractor.py` | Извлечение MAR, EAR, Nose-Chin, yaw, roll через MediaPipe |
| `Polytope1D` | `polytope.py` | Одномерная политопная фильтрация (mode: above/below) |
| `Polytope2D` | `polytope.py` | Двумерная политопная фильтрация (MAR + Nose-Chin) |
| `PERCLOSCalculator` | `perclos.py` | Расчет PERCLOS в скользящем окне 30 секунд |
| `CriticalityEvaluator` | `criticality.py` | Интегральная оценка критичности (0-3) |
| `Calibrator` | `calibrator.py` | 30-секундная калибровка, расчет индивидуальных порогов |
| `DatabaseManager` | `database.py` | SQLite: профили водителей, данные мониторинга, события |
| `Visualizer` | `visualizer.py` | Отрисовка ключевых точек, метрик, цветовой рамки |
| `ReportGenerator` | `reporter.py` | Генерация текстовых отчетов и графиков |

### 3.3. Ключевые архитектурные решения

1. **Пропуск кадров:** 30 FPS → 15 FPS (снижение нагрузки на CPU на 50%)
2. **Privacy by Design:** сохранение только числовых метрик, а не изображений
3. **Локальная обработка:** все вычисления на устройстве пользователя, данные не передаются по сети
4. **Адаптивная калибровка:** 30 секунд для каждого нового водителя
5. **Политопная фильтрация:** использование `collections.deque` для скользящего окна (O(1) на кадр)

---

## 4. Установка и запуск

### 4.1. Требования

- Python 3.11 или выше
- USB-веб-камера (RGB)
- ОС: Windows 10/11, Ubuntu 20.04+, macOS 11+

### 4.2. Установка

```bash
git clone https://github.com/StScythe/driver-monitoring.git
cd driver-monitoring

python -m venv venv
source venv/bin/activate      # Linux/macOS
# venv\Scripts\activate       # Windows

pip install -r requirements.txt

4.3. Запуск
bash
python scripts/run.py
4.4. Интерфейс управления
text
======================================
СИСТЕМА МОНИТОРИНГА СОСТОЯНИЯ ВОДИТЕЛЯ
======================================

Главное меню:
1. Идентификация водителя
2. Запуск мониторинга
3. Генерация отчета
4. Выход
Порядок работы:

Выбрать 1. Идентификация водителя — при первом запуске проводится 30-секундная калибровка.

Выбрать 2. Запуск мониторинга — остановка по q или ESC.

Выбрать 3. Генерация отчета — текстовый отчет и графики.

5. Результаты тестирования
5.1. Приемочные тесты (unittest)
Разработано и выполнено 18 автоматизированных приемочных тестов:

Категория	Количество	Результат
Математические формулы (MAR, EAR, расстояние)	6	✅ Пройдено
Политопная логика (1D, 2D)	4	✅ Пройдено
PERCLOS (расчет, уровни, сброс)	3	✅ Пройдено
Оценка критичности (матрица сочетаний)	3	✅ Пройдено
База данных (сохранение, загрузка)	2	✅ Пройдено
Всего	18	✅ 18 пройдено
5.2. Тестирование производительности
Стенд: Intel Core i5-9600K (6 ядер), 16 ГБ ОЗУ, USB-камера 720p @ 30 FPS

Метрика	Целевое значение	Измеренное значение	Статус
Частота обработки (FPS)	≥ 15	15.3	✅
Задержка обработки кадра	≤ 200 мс	≈ 0.1 мс	✅
Потребление ОЗУ	≤ 1 ГБ	≈ 250 МБ	✅
5.3. Результаты детекции на тестовом сеансе
Тестовый сеанс: 86.9 секунды, водитель с ID=2 (калибровка выполнена)

Система детекции	Покадровых событий	Политопных событий	Снижение ложных
Зевок (MAR + расстояние)	1	1	0%
Сонливость (EAR)	81	1	98.8%
Отвлечение внимания (|yaw| > 30°)	39	1	97.4%
Наклон головы (|roll| > 20°)	17	3	82.4%
Выводы:

Политопная фильтрация эффективно отсекает кратковременные события (моргания, быстрые повороты)

Истинные длительные события (зевок 1.8 с, сонливость 4.8 с) успешно детектируются

Снижение ложных срабатываний: 97-99% для сонливости и отвлечения

5.4. Графический отчет
По окончании мониторинга система генерирует 5 графиков:

EAR — динамика открытости глаз, выделение интервалов сонливости

MAR — динамика открытости рта, выделение интервалов зевков

Nose-Chin — динамика расстояния нос-подбородок

Yaw / Roll — поза головы, выделение отвлечения и наклона

PERCLOS — процент времени с закрытыми глазами, уровни 10%/20%/28%

Пример вывода статистики:

text
============================================================
СТАТИСТИКА МОНИТОРИНГА СОСТОЯНИЯ ВОДИТЕЛЯ
============================================================
Общее время анализа: 87.3 сек

ОБНАРУЖЕНО СОБЫТИЙ (политопный метод):
  Зевки:                         1
  Отвлечение внимания:           1
  Наклоны головы:                3
  Сонливость:                    1

ЭФФЕКТИВНОСТЬ ПОЛИТОПНОЙ ФИЛЬТРАЦИИ:
  Зевки:               снижение на 0.0%
  Отвлечение:          снижение на 97.4%
  Наклоны:             снижение на 82.4%
  Сонливость:          снижение на 98.8%

РАСПРЕДЕЛЕНИЕ УРОВНЕЙ КРИТИЧНОСТИ:
  NORMAL         32.3 сек ( 37.0%)
  WARNING         0.7 сек (  0.8%)
  DANGER         32.7 сек ( 37.5%)
  CRITICAL       21.6 сек ( 24.7%)
6. Выявленные технические ограничения
В ходе разработки и тестирования MVP выявлены следующие ограничения:

Ограничение	Описание	Влияние на систему
Нестабильность оценки угла pitch	Метод solvePnP на монокулярном изображении выдаёт значения pitch, стремящиеся к ±180° при нейтральном положении головы	Угол pitch исключён из анализа. Для детекции используются только yaw и roll
Чувствительность к освещению	MediaPipe работает корректно при естественном освещении, но при низкой освещенности точность детекции падает	Для ночных условий требуется ИК-камера (перспектива развития)
Ракурс съемки	Требуется фронтальное расположение лица. При повороте головы >45° детекция нестабильна	Система временно отключает детекцию при экстремальных ракурсах
Очки и маски	MediaPipe корректно работает с очками, но маски, закрывающие рот, делают невозможным расчет MAR	При детекции маски система использует только EAR для оценки сонливости
7. Этический аспект (ФЗ-152 «О персональных данных»)
7.1. Текущая реализация MVP
Требование ФЗ-152	Реализация в MVP
Локальная обработка	Все вычисления на ПК пользователя, данные не передаются по сети
Отсутствие сохранения изображений	Кадры обрабатываются в реальном времени и отбрасываются
Минимизация данных	В SQLite сохраняются только числовые метрики (EAR, MAR, yaw, roll, PERCLOS)
Отсутствие передачи третьим лицам	Нет сетевых вызовов, нет облачных сервисов
7.2. Принцип Privacy by Design
Архитектура системы с самого начала проектировалась с учетом приватности:

Нет сохранения видеопотока

Нет передачи данных по сети

Локальная база данных SQLite

Индивидуальная калибровка без передачи данных на сервер

7.3. Ограничения текущей версии
Калибровка: требует 30 секунд нахождения водителя в кадре с открытыми глазами и закрытым ртом.

Освещение: система чувствительна к экстремально низкой освещенности (требуется ИК-камера для ночной работы).

Ракурс: требует фронтального расположения лица в кадре.

Очки и маски: MediaPipe корректно работает с очками, но маски снижают точность детекции.

8. Структура проекта
text
driver-monitoring/
├── scripts/
│   └── run.py                      # Точка входа
│
├── src/dms/                        # MVP-ядро
│   ├── config.py                   # Конфигурация, загрузка коэффициентов
│   ├── extractor.py                # Извлечение метрик (MediaPipe + NumPy)
│   ├── polytope.py                 # 1D и 2D политопы
│   ├── perclos.py                  # PERCLOSCalculator
│   ├── criticality.py              # CriticalityEvaluator
│   ├── calibrator.py               # Calibrator (30 сек)
│   ├── database.py                 # DatabaseManager (SQLite)
│   ├── visualizer.py               # Visualizer (OpenCV)
│   ├── reporter.py                 # ReportGenerator (pandas, matplotlib)
│   ├── analyzer.py                 # FatigueAnalyzer
│   └── system.py                   # FatigueMonitoringSystem
│
├── src/tests/                      # Приемочные тесты (unittest)
│   ├── test_formulas.py
│   ├── test_polytope.py
│   ├── test_perclos.py
│   ├── test_criticality.py
│   └── test_database.py
│
├── data/
│   └── scientific_coefficients.json   # Коэффициенты от научного модуля
│
├── requirements.txt
└── README.md
9. Зависимости
Библиотека	Версия	Назначение
opencv-python	4.12.0.88	Захват видео, визуализация, solvePnP
mediapipe	0.10.14	Детекция лица и 468 ключевых точек
numpy	2.2.6	Векторизованные вычисления
pandas	2.3.3	Обработка данных для отчетов
matplotlib	3.10.7	Построение графиков
Дата: 2026-06-05
Автор: Волков Д.В.